<a href="https://colab.research.google.com/github/adwaith-santhosh/ict-miniproject/blob/main/chicago_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import time
from datetime import datetime
import numpy as np
import pandas as pd

In [ ]:
DATA_PATH='/content/Crimes_-_2015_20260702.xls'
OUT_DIR='Outputs'
INTERVAL_SECONDS=20
MAX_RUNS=3

In [ ]:
def extract():
  df=pd.read_csv(DATA_PATH)
  df_len=len(df)
  raw_file_path=os.path.join(OUT_DIR,'raw_crimes_data.csv')
  df.to_csv(raw_file_path,index=False)
  print(f"raw data saved to {raw_file_path}")
  print(f"Extracted {df_len} rows")
  return df

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
os.path.exists(OUT_DIR)

True

In [ ]:
extract()

raw data saved to Outputs/raw_crimes_data.csv
Extracted 264902 rows


,ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,...,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location
0,10460641,HZ199559,12/31/2015 11:59:00 PM,015XX N KEDZIE AVE,0890,THEFT,FROM BUILDING,RESIDENCE PORCH/HALLWAY,False,False,...,26.0,23.0,06,NaN,NaN,2015,2018 Feb 09 03:44:29 PM,NaN,NaN,NaN
1,10365064,HZ100370,12/31/2015 11:59:00 PM,075XX S EMERALD AVE,1320,CRIMINAL DAMAGE,TO VEHICLE,STREET,False,False,...,17.0,68.0,14,1172605.0,1854931.0,2015,2018 Feb 10 03:50:01 PM,41.757367,-87.642993,POINT (-87.642992854 41.757366519)
2,10364662,HZ100006,12/31/2015 11:55:00 PM,079XX S STONY ISLAND AVE,0430,BATTERY,AGGRAVATED: OTHER DANG WEAPON,STREET,False,False,...,8.0,45.0,04B,1188223.0,1852840.0,2015,2018 Feb 10 03:50:01 PM,41.751270,-87.585822,POINT (-87.585822373 41.751270452)
3,10364683,HZ100002,12/31/2015 11:50:00 PM,037XX N CLARK ST,0460,BATTERY,SIMPLE,SIDEWALK,True,False,...,44.0,6.0,08B,1167786.0,1925033.0,2015,2018 Feb 10 03:50:01 PM,41.949837,-87.658635,POINT (-87.658635101 41.949837364)
4,10364740,HZ100010,12/31/2015 11:50:00 PM,024XX W FARGO AVE,0820,THEFT,$500 AND UNDER,APARTMENT,False,False,...,50.0,2.0,06,1158878.0,1949369.0,2015,2018 Feb 10 03:50:01 PM,42.016804,-87.690709,POINT (-87.690708662 42.016804165)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
264897,10828511,HZ554879,01/01/2015 12:00:00 AM,031XX W BELDEN AVE,0281,CRIM SEXUAL ASSAULT,NON-AGGRAVATED,APARTMENT,False,True,...,35.0,22.0,02,NaN,NaN,2015,2021 Sep 07 03:41:02 PM,NaN,NaN,NaN
264898,9982032,HY171603,01/01/2015 12:00:00 AM,007XX E 84TH ST,1153,DECEPTIVE PRACTICE,FINANCIAL IDENTITY THEFT OVER $ 300,APARTMENT,False,False,...,6.0,44.0,11,1182571.0,1849441.0,2015,2018 Feb 10 03:50:01 PM,41.742076,-87.606639,POINT (-87.606639129 41.742076111)
264899,10951142,JA270602,01/01/2015 12:00:00 AM,070XX S PAXTON AVE,1153,DECEPTIVE PRACTICE,FINANCIAL IDENTITY THEFT OVER $ 300,RESIDENCE,False,False,...,5.0,43.0,11,NaN,NaN,2015,2017 May 20 03:49:22 PM,NaN,NaN,NaN
264900,11013009,JA342912,01/01/2015 12:00:00 AM,076XX S INGLESIDE AVE,0281,CRIM SEXUAL ASSAULT,NON-AGGRAVATED,RESIDENCE,False,False,...,8.0,69.0,02,NaN,NaN,2015,2017 Jul 11 03:48:22 PM,NaN,NaN,NaN


In [ ]:
def transform(df):
  crimes_samples=df.sample(n=10000,random_state=46).reset_index(drop=True)
  crimes_samples['Extracted at']=datetime.now()
  crimes_samples.drop_duplicates(inplace=True)
  crimes_samples['Date']=pd.to_datetime(crimes_samples['Date'])
  crimes_samples['Crime_Date']=crimes_samples['Date'].dt.date
  crimes_samples['Crime_Time']=crimes_samples['Date'].dt.time
  crimes_samples[['Block_Number','Direction','Street']]=crimes_samples['Block'].str.extract(r'(\S+)\s+([NSEW])\s+(.+)')
  crimes_samples.drop(columns=['Block','Date','Updated On','X Coordinates','Y Coordinates','Year','Location'],errors = 'ignore', inplace=True)
  crimes_samples['Location Description']=crimes_samples['Location Description'].fillna('UNKNOWN')
  crimes_samples.dropna(subset=['Ward','Community Area','Latitude','Longitude'],inplace=True)
  print(f"Transformed {len(crimes_samples)} rows")
  return crimes_samples

In [ ]:
def load(df):
  timestamp=datetime.now().strftime("%Y%m%d_%H%M%S")
  cleaned_path=os.path.join(OUT_DIR,f"cleaned_crimes_{timestamp}.csv")
  df.to_csv(cleaned_path,index=False)
  summary=pd.DataFrame({
      'metric': ['Total Records','Total Arrests','Domestic Crimes'],
      'value':[len(df),df["Arrest"].sum(),df['Domestic'].sum()]
  })

  filename=datetime.now().strftime("%Y%m%d_%H%M%S")
  load_path=f"{OUT_DIR}/summary_{filename}.csv"

  summary.to_csv(load_path,index=False)
  print(f"Loaded{len(df)} rows")

In [ ]:
def run_pipeline():
  df=extract()
  cleaned_data=transform(df)
  load(cleaned_data)
  print("pipeline completed")

In [ ]:
def schedule_pipeline():
  run_count=0
  while run_count<MAX_RUNS:
    run_pipeline()
    run_count+=1

    if run_count<MAX_RUNS:
      print(f"next run will happen after {INTERVAL_SECONDS}")
      time.sleep(INTERVAL_SECONDS)
  print("All schedule runs are completed")

if __name__=='__main__':
  schedule_pipeline()

raw data saved to Outputs/raw_crimes_data.csv
Extracted 264902 rows


/tmp/ipykernel_6421/3911668217.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  crimes_samples['Date']=pd.to_datetime(crimes_samples['Date'])


Transformed 9749 rows
Loaded9749 rows
pipeline completed
next run will happen after 20
raw data saved to Outputs/raw_crimes_data.csv
Extracted 264902 rows


/tmp/ipykernel_6421/3911668217.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  crimes_samples['Date']=pd.to_datetime(crimes_samples['Date'])


Transformed 9749 rows
Loaded9749 rows
pipeline completed
next run will happen after 20
raw data saved to Outputs/raw_crimes_data.csv
Extracted 264902 rows


/tmp/ipykernel_6421/3911668217.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  crimes_samples['Date']=pd.to_datetime(crimes_samples['Date'])


Transformed 9749 rows
Loaded9749 rows
pipeline completed
All schedule runs are completed
